# CARRA reanalysis atmosphere

Builds atmosphere profiles for a clear-sky thermal simulation at Kiruna from CARRA (Copernicus Arctic Regional Reanalysis, 2.5 km grid), which resolves the Arctic better than ERA5. CARRA is passed through the same `era5_atmosphere=` interface as ERA5 ({doc}`/guide/atmosphere`, {doc}`era5_atmosphere`).

```{note}
The CARRA file is a local extract on `/projekt_agmwend` and is not distributed with pyRadtran. This notebook is not executed for the book and has no stored outputs; run it with your own CARRA or ERA5 pressure-level data.
```

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import pyradtran
from pyradtran import load_config

logging.getLogger('pyradtran').setLevel(logging.CRITICAL)
Path("work").mkdir(exist_ok=True)

kiruna_lat, kiruna_lon = 67.84889, 20.3027

## Loading CARRA data

CARRA pressure-level data carry relative humidity `r` (%) instead of specific humidity `q`, which the atmosphere writer needs. `q` is computed from `r`, `t` and the pressure with the saturation vapour pressure over water (Tetens formula with the IFS constants). Below about −20 °C reanalyses usually define `r` partly with respect to ice, so this `q` is then an upper bound. The pressure coordinate `plev` and its unit (Pa or hPa) are recognised by pyRadtran.

In [ ]:
ds_carra = xr.open_dataset("/projekt_agmwend/data_raw/COMPEX-EC/04_ERA5/CARRA_pressure_levels/CARRA_point_optimized.zarr")
ds_kiruna = ds_carra.sel(lat=kiruna_lat, lon=kiruna_lon, method='nearest').load()
ds_kiruna = ds_kiruna.rename({'lat': 'latitude', 'lon': 'longitude'})

# Specific humidity from relative humidity
t = ds_kiruna['t']                                          # K
p_pa = ds_kiruna['plev'] * (1 if float(ds_kiruna['plev'].max()) > 2000 else 100)
e_sat = 611.21 * np.exp(17.502 * (t - 273.16) / (t - 32.19))  # Pa, over water
e = ds_kiruna['r'] / 100 * e_sat
ds_kiruna['q'] = 0.622 * e / (p_pa - 0.378 * e)             # kg/kg
ds_kiruna['q'].attrs['units'] = 'kg kg-1'
ds_kiruna

## Running simulations

Clear-sky thermal irradiance (2.5–100 µm) from the surface to 19 km at every CARRA time step. The surface is snow (emissivity 0.99, i.e. albedo 0.01) at a fixed 248.4 K.

In [ ]:
cfg = load_config()
cfg.simulation_defaults.rte_solver = "disort"
cfg.simulation_defaults.mol_abs_param = "reptran coarse"
cfg.simulation_defaults.source = "thermal"
cfg.simulation_defaults.wavelength_nm = [2500, 100000]
cfg.simulation_defaults.integrate_wavelength = True
cfg.simulation_defaults.albedo_value = 0.01                 # emissivity 0.99
cfg.simulation_defaults.surface_temperature_k = 248.4       # assumption
cfg.simulation_defaults.output_columns = ["zout", "edn", "eup", "enet"]
cfg.execution.max_workers = 8
config_path = Path("work/carra_thermal.yaml")
cfg.to_yaml(config_path)

n_times = ds_kiruna.sizes['time']
ds = xr.Dataset(
    coords={
        'time': ds_kiruna['time'].values,
        'altitude': ('altitude', np.arange(0, 20, 1.0)),    # output levels (km)
    },
    data_vars={
        'latitude': ('time', np.full(n_times, kiruna_lat)),
        'longitude': ('time', np.full(n_times, kiruna_lon)),
    },
)

ds_sim = ds.pyradtran.run(
    config_path=config_path,
    era5_atmosphere=ds_kiruna,
    save_to_file=False,
    show_progress=False,
)
ds_sim

## Distance to the nearest CARRA grid point

Great-circle distance between Kiruna and the CARRA grid point used.

In [ ]:
def haversine(lon1, lat1, lon2, lat2):
    """Great-circle distance in km."""
    lon1, lat1, lon2, lat2 = map(np.deg2rad, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


d = haversine(kiruna_lon, kiruna_lat, float(ds_kiruna.longitude), float(ds_kiruna.latitude))
print(f"Distance to the nearest CARRA point: {d:.2f} km")

## Results

CARRA temperature (top) and simulated net thermal irradiance $E^\downarrow - E^\uparrow$ (bottom, W m⁻²).

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ds_kiruna.t.plot.pcolormesh(x='time', y='plev', ax=ax[0], cmap='inferno', robust=True, yincrease=False,
                            cbar_kwargs={'label': 'Temperature (K)'})
ax[0].set_yscale('log')
ax[0].set_ylabel('Pressure')

ds_sim.enet.plot(x='time', y='altitude', ax=ax[1], cmap='viridis', robust=True,
                 cbar_kwargs={'label': 'Net thermal irradiance (W/m²)'})
ax[1].set_ylabel('Altitude (km)')
for a in ax:
    a.set_title('')
plt.tight_layout()